# Retrain the captioning model on MS COCO (Google Colab GPU)

Same CNN + LSTM design as the 2021 model (`model_9.h5`), trained on **MS COCO 2017** (~118,000 images,
~590,000 captions) instead of Flickr8k (8,000 images). More images and a bigger vocabulary
(~9,000 words instead of 1,848) give more accurate, more varied captions.

**Before you start:** `Runtime → Change runtime type → T4 GPU`, then run the cells in order.

| Step | Time on a free T4 (approx.) |
|---|---|
| Download COCO (19 GB) | 10–20 min |
| ResNet50 features for 117k images | 20–40 min (cached on Google Drive) |
| Training, per epoch | 5–15 min (usually stops after 10–20 epochs) |

Everything important is saved to **Google Drive** (`MyDrive/vcav_coco`). If Colab disconnects,
just run all cells again: feature extraction and training **resume** where they stopped.

In [ ]:
!nvidia-smi -L
from google.colab import drive
drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/vcav_coco'      # results are kept here

## 1. Get the project code

In [ ]:
%cd /content
!git clone -q https://github.com/vkotta9/video-captioning-assistive-vision.git || (cd video-captioning-assistive-vision && git pull -q)
%cd /content/video-captioning-assistive-vision
!pip install -q nltk
import tensorflow as tf; print('TensorFlow', tf.__version__, '| GPU:', tf.config.list_physical_devices('GPU'))

## 2. Download MS COCO 2017 (train images + captions)
The 18 GB image zip is **not** extracted: the training script reads images straight from the zip.

In [ ]:
%cd /content
!wget -q -c --show-progress http://images.cocodataset.org/annotations/annotations_trainval2017.zip
!unzip -q -o annotations_trainval2017.zip annotations/captions_train2017.json
!wget -q -c --show-progress http://images.cocodataset.org/zips/train2017.zip
!ls -lh train2017.zip annotations/captions_train2017.json

## 3. GloVe word vectors (50-d, as in 2021)

In [ ]:
%cd /content
import os
if not os.path.exists('glove.6B.50d.txt'):
    !wget -q -c --show-progress https://nlp.stanford.edu/data/glove.6B.zip
    !unzip -q -o glove.6B.zip glove.6B.50d.txt
GLOVE = '--glove /content/glove.6B.50d.txt' if os.path.exists('glove.6B.50d.txt') else ''
print('GloVe:', GLOVE or 'not available - the embedding will be learned instead (also fine)')

## 4. Train
* `--threshold 10`: keep words seen more than 10 times (same rule as 2021).
* `--max-len 35`: same maximum caption length as the original model.
* Validation loss is checked after every epoch; the best model is saved as `best.keras`, the learning rate is
  halved when it stops improving, and training stops after 3 epochs without improvement.

In [ ]:
%cd /content/video-captioning-assistive-vision
!python training/train.py \
    --captions /content/annotations/captions_train2017.json \
    --images /content/train2017.zip \
    --out {OUT} {GLOVE} \
    --epochs 20 --batch-size 1024

In [ ]:
import pandas as pd
h = pd.read_csv(f'{OUT}/history.csv'); print(h.to_string(index=False))
h.plot(x='epoch', y=['loss', 'val_loss'], marker='o', title='Training vs validation loss');

## 5. Compare with the original 2021 model (BLEU on 1,000 held-out COCO images)

In [ ]:
%cd /content/video-captioning-assistive-vision
!python training/evaluate_bleu.py \
    --captions /content/annotations/captions_train2017.json --images /content/train2017.zip \
    --models model_weights/model_9.h5 {OUT}/model_weights/best.keras \
    --test-list {OUT}/storage/test_images.txt --limit 1000 --beam 3 --show 8

## 6. Look at some captions

In [ ]:
%cd /content/video-captioning-assistive-vision
import sys, zipfile, io, random
sys.path.insert(0, '/content/video-captioning-assistive-vision')
import matplotlib.pyplot as plt
from PIL import Image
from caption import Captioner
old = Captioner('model_weights/model_9.h5')
new = Captioner(f'{OUT}/model_weights/best.keras', encoder=old.encoder)
z = zipfile.ZipFile('/content/train2017.zip')
test = open(f'{OUT}/storage/test_images.txt').read().split()
for name in random.sample(test, 6):
    img = Image.open(io.BytesIO(z.read('train2017/' + name))).convert('RGB')
    img.save('/tmp/x.jpg')
    plt.figure(figsize=(5, 4)); plt.imshow(img); plt.axis('off')
    plt.title(f"2021:  {old.caption_image('/tmp/x.jpg', 3)}\nCOCO:  {new.caption_image('/tmp/x.jpg', 3)}", fontsize=9)
    plt.show()

## 7. Download the retrained model
This makes `retrained_coco.zip` (~15 MB). Unzip it **inside the project folder** on your laptop, so you get
`video-captioning-assistive-vision/retrained_coco/model_weights/best.keras` and `.../retrained_coco/storage/`. Then run:

```
python live_caption.py --model retrained_coco/model_weights/best.keras
```

In [ ]:
!rm -rf /content/retrained_coco /content/retrained_coco.zip && mkdir -p /content/retrained_coco/model_weights
!cp {OUT}/model_weights/best.keras /content/retrained_coco/model_weights/
!cp -r {OUT}/storage /content/retrained_coco/ && rm -rf /content/retrained_coco/storage/test_features*
!cp {OUT}/history.csv /content/retrained_coco/
!cd /content && zip -q -r retrained_coco.zip retrained_coco && ls -lh retrained_coco.zip
from google.colab import files
files.download('/content/retrained_coco.zip')